# schedule

> the standing instruction, its clock, and the tick that turns a due one into queued work

In [ ]:
#| default_exp schedule

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

A schedule is one row. It holds an instruction in your own words, when it repeats, what list it
reads, what an unattended run of it may do, and the conversation it continues. `tick` is what a
cron entry calls. It turns every due schedule into queued jobs and returns.

It does not run anything. The queue does that. What the queue's handler does is
[run](06_run.ipynb)'s problem. A fire kept in the queue rather than in the tick is retryable,
leaseable and recorded like any other job.

Two ticks racing cannot double-fire. One `BEGIN IMMEDIATE` covers reading a schedule and advancing
it, so the two serialise. Each fire is then enqueued under the key `<schedule>:<fire time>`, which
the queue's unique index refuses twice. That is why there is no lock table here.

In [ ]:
#| export
from fastcore.all import AttrDict, L, ifnone, patch, xdumps
from pobblebonk.clock import CATCHUP, due_fires, late, next_fire, secs, valid_cron, valid_tz
from pobblebonk.core import Store, clip, mkid, now, row

In [ ]:
#| export
FIRE = 'pob.fire'                # the queue kind every schedule fire uses
POLICIES = ('read', 'write')     # what an unattended fire may do; there is no `ask` when nobody is there
MAX_PROMPT = 8_000               # chars of a standing instruction

`policy` has two values on purpose. An unattended fire has nobody to answer a permission request.
A third value meaning "ask me" would be a request into an empty room. `read` refuses writes and
says so in the note. `write` is you deciding in advance. This is the setting that keeps "check my
cart on Wednesday" from placing an order.

In [ ]:
#| export
class Schedules(Store):
    "Standing instructions and their clocks."
    name = 'pob_schedules'
    cols = dict(id=str, name=str, prompt=str, every=float, cron=str, tz=str, next_run=float,
                last_run=float, catchup=str, needs=str, blocked=str, policy=str, session=str,
                payload=str, paused=int, created_at=float, updated_at=float)
    idx = ('CREATE INDEX IF NOT EXISTS pob_sched_due ON pob_schedules(next_run) WHERE paused=0',
           'CREATE UNIQUE INDEX IF NOT EXISTS pob_sched_name ON pob_schedules(name)')

    def __repr__(self): return f'Schedules({len(self.all())} of them, {len(self.due())} due)'

## Writing one down

In [ ]:
#| export
@patch
def add(self:Schedules,
        name:str,               # what you call it; unique, and what a note is titled with
        prompt:str,             # the instruction, in your own words
        every=None,             # a duration: `'30m'`, `'6h'`, `'1w'`, or seconds
        cron:str=None,          # or a five-field cron expression, instead of `every`
        tz:str=None,            # the zone `cron` is read in, at every fire
        catchup:str='once',     # what to do with fires missed while the machine was off
        needs:str=None,         # a list topic this cannot run without
        policy:str='read',      # `read` refuses writes on an unattended run; `write` allows them
        payload:dict=None,      # anything the handler wants that is not the prompt
        start:float=None,       # first fire; the next natural one when unset
        at:float=None
) -> AttrDict:
    'Write down a standing instruction. What cannot be read is refused now, not at 5pm.'
    at = ifnone(at, now())
    if bool(every) == bool(cron): raise ValueError('a schedule needs exactly one of `every` or `cron`')
    if cron and not valid_cron(cron): raise ValueError(f'not a cron expression: {cron!r}')
    if tz and not valid_tz(tz): raise ValueError(f'not a time zone: {tz!r}')
    if catchup not in CATCHUP: raise ValueError(f'catchup must be one of {CATCHUP}: {catchup!r}')
    if policy not in POLICIES: raise ValueError(f'policy must be one of {POLICIES}: {policy!r}')
    if not str(prompt or '').strip(): raise ValueError('a schedule needs a prompt: it is all the fire gets')
    ev = secs(every) if every else None
    nxt = ifnone(start, next_fire(cron, tz, at) if cron else at + ev)
    r = dict(id=mkid(), name=str(name), prompt=clip(prompt, MAX_PROMPT), every=ev, cron=cron, tz=tz,
             next_run=nxt, last_run=None, catchup=catchup, needs=needs, blocked=None, policy=policy,
             session=None, payload=xdumps(payload), paused=0, created_at=at, updated_at=at)
    with self.txn(): self.t.insert(r)
    return row(r)

In [ ]:
#| export
@patch
def get(self:Schedules, which:str) -> AttrDict:
    'One schedule by id or by name.'
    return self.one('SELECT * FROM pob_schedules WHERE id=? OR name=?', [which, which])

@patch
def all(self:Schedules, paused:bool=None) -> L:
    'Every schedule, soonest first.'
    w, a = ('WHERE paused=? ', [int(paused)]) if paused is not None else ('', [])
    return self.q(f'SELECT * FROM pob_schedules {w}ORDER BY next_run', a)

@patch
def due(self:Schedules, at:float=None, limit:int=50) -> L:
    'Schedules whose next fire has arrived.'
    return self.q('SELECT * FROM pob_schedules WHERE paused=0 AND next_run<=? ORDER BY next_run '
                  'LIMIT ?', [ifnone(at, now()), limit])

@patch
def _set(self:Schedules, sched_id:str, at:float=None, **kw) -> AttrDict:
    'Update named columns on one schedule.'
    kw['updated_at'] = ifnone(at, now())
    sets = ', '.join(f'{k}=?' for k in kw)
    with self.txn():
        self.db.q(f'UPDATE pob_schedules SET {sets} WHERE id=?', [*kw.values(), sched_id])
    return self.get(sched_id)

@patch
def pause(self:Schedules, which:str) -> AttrDict:
    'Stop it firing, keeping the row and its history.'
    return self._set(self.get(which)['id'], paused=1)

@patch
def resume(self:Schedules, which:str, at:float=None) -> AttrDict:
    'Start it again from the next natural fire, and forget that it was ever blocked.'
    s, at = self.get(which), ifnone(at, now())
    nxt = next_fire(s['cron'], s['tz'], at) if s['cron'] else at + max(s['every'] or 1, 1)
    return self._set(s['id'], paused=0, blocked=None, next_run=nxt, at=at)

@patch
def remember(self:Schedules, which:str, session:str) -> AttrDict:
    'Keep the conversation this schedule runs in, so tomorrow can talk about today.'
    return self._set(self.get(which)['id'], session=session)

@patch
def drop(self:Schedules, which:str) -> bool:
    'Delete one schedule. Notes it already left stay where they are.'
    with self.txn():
        return bool(self.db.q('DELETE FROM pob_schedules WHERE id=? OR name=? RETURNING id',
                              [which, which]))

## The tick

One transaction per schedule, and the schedule is read again inside it. A tick that lost the race
finds `next_run` already moved on and does nothing.

In [ ]:
#| export
@patch
def tick(self:Schedules,
         q,                  # the `Queue` fires are enqueued on
         at:float=None,      # when this tick is happening
         inbox=None,         # where a blocked schedule says what it is waiting for
         have=None,          # `(topic) -> int` open items, for a schedule with `needs`
         limit:int=50        # schedules looked at in one tick
) -> AttrDict:
    'Turn every due schedule into queued fires. This is what a cron entry calls.'
    at = ifnone(at, now())
    fired, blocked, freed = L(), L(), L()
    for s in self.due(at, limit):
        with self.txn():
            cur = self.get(s['id'])
            # another tick may have taken this one between the read above and the lock here
            if cur is None or cur['paused'] or cur['next_run'] > at: continue
            times, nxt = due_fires(cur, at, cur['catchup'] or 'once')
            short = bool(cur['needs']) and have is not None and not have(cur['needs'])
            if short:
                times = []
                if cur['blocked'] != cur['needs']:
                    blocked.append(cur['name'])
                    if inbox is not None:
                        inbox.note('blocked', f"{cur['name']} needs your {cur['needs']} list",
                                   f"Nothing is on the {cur['needs']} list, so this had nothing to "
                                   f"work from. Add to it and the next fire runs.",
                                   ref=cur['id'], at=at)
            elif cur['blocked']: freed.append(cur['name'])
            for t in times:
                fired.append(q.enqueue(FIRE, dict(schedule=cur['id'], fire_at=t, late=late(t, at)),
                                       key=f"{cur['id']}:{int(t)}", at=at))
            self._set(cur['id'], next_run=nxt, blocked=(cur['needs'] if short else None),
                      last_run=(times[-1] if times else cur['last_run']), at=at)
    return AttrDict(fired=fired, blocked=blocked, freed=freed)

The 5pm roundup. Registered on a Monday morning, it is not due until 5pm:

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
from pobblebonk.core import vault
from pobblebonk.inbox import Inbox
from pobblebonk.lists import Lists
from pobblebonk.queue import Queue

kol = ZoneInfo('Asia/Kolkata')
def when(d, h, m=0): return datetime(2026, 8, d, h, m, tzinfo=kol).timestamp()

db = vault()
sch, q, ib, ls = Schedules(db), Queue(db), Inbox(db), Lists(db)
sch.add('git roundup', 'Roundup my commits today and say how it compares with last week.',
        cron='0 17 * * *', tz='Asia/Kolkata', at=when(24, 9))

test_eq(len(sch.due(when(24, 9))), 0)
test_eq(len(sch.due(when(24, 17, 1))), 1)

The tick at one minute past five queues exactly one fire, and moves the clock to tomorrow:

In [ ]:
out = sch.tick(q, at=when(24, 17, 1))
test_eq(len(out.fired), 1)
test_eq(q.pending(FIRE), 1)
test_eq(datetime.fromtimestamp(sch.get('git roundup')['next_run'], kol).day, 25)

A second tick in the same minute finds nothing, and even a tick told it is still 17:01 cannot
queue that fire again:

In [ ]:
test_eq(len(sch.tick(q, at=when(24, 17, 2)).fired), 0)
test_eq(q.pending(FIRE), 1)

## A schedule that needs a list

The news digest cannot guess your interests, and at 7am there is nobody to ask. So it does not
run: it says what it is missing, once, and starts again on its own when the list fills.

In [ ]:
db3 = vault()
s3, q3, ib3, ls3 = Schedules(db3), Queue(db3), Inbox(db3), Lists(db3)
s3.add('news', 'Give me the news that matters, based on my interests.',
       cron='0 7 * * *', tz='Asia/Kolkata', needs='interests', at=when(24, 18))
have = lambda topic: ls3.count(topic)

out = s3.tick(q3, at=when(25, 7, 1), inbox=ib3, have=have)
test_eq(out.blocked, ['news'])
test_eq(q3.pending(FIRE), 0)                                  # nothing was queued
test_eq(ib3.notes(kind='blocked')[0]['title'], 'news needs your interests list')

The next morning says nothing, because it has already said it:

In [ ]:
test_eq(s3.tick(q3, at=when(26, 7, 1), inbox=ib3, have=have).blocked, [])
test_eq(ib3.count('blocked'), 1)

You add two interests. Nobody resumes anything: the next fire simply runs.

In [ ]:
ls3.add('interests', 'sanskrit grammar')
ls3.add('interests', 'sqlite internals')
out = s3.tick(q3, at=when(27, 7, 1), inbox=ib3, have=have)
test_eq(out.freed, ['news'])
test_eq(len(out.fired), 1)
test_eq(s3.get('news')['blocked'], None)

## Catching up

Nine days of a shut laptop, and `once` is the default for a reason:

In [ ]:
db2 = vault()
s2, q2 = Schedules(db2), Queue(db2)
s2.add('daily', 'roundup', cron='0 17 * * *', tz='Asia/Kolkata', start=when(15, 17))
s2.add('every fire counts', 'file one report per day', cron='0 18 * * *', tz='Asia/Kolkata',
       catchup='all', start=when(15, 18))

out = s2.tick(q2, at=when(24, 9))
test_eq(len(out.fired), 1 + 9)      # one for the roundup, nine for the one that wants them all

What a fire carries is small on purpose: which schedule, which fire, and whether it is late. The
handler reads the rest out of the vault when it runs.

In [ ]:
p = q2.jobs(kind=FIRE)[0].payload
test_eq(sorted(p), ['fire_at', 'late', 'schedule'])
test_eq(p['late'], True)

## Refusing what cannot be read

In [ ]:
test_fail(lambda: sch.add('bad', 'x', cron='every friday'), contains='not a cron')
test_fail(lambda: sch.add('bad', 'x', cron='0 17 * * *', tz='Mars/Olympus'), contains='not a time zone')
test_fail(lambda: sch.add('bad', 'x'), contains='exactly one')
test_fail(lambda: sch.add('bad', 'x', every='1h', cron='0 17 * * *'), contains='exactly one')
test_fail(lambda: sch.add('bad', '  ', every='1h'), contains='needs a prompt')
test_fail(lambda: sch.add('bad', 'x', every='1h', policy='ask'), contains='policy must be')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()